<a href="https://colab.research.google.com/github/Amritpalkaur7/Sign-language-translator/blob/main/sign_language_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mount Google Drive in Colab**
it is used to connect Google Drive with Google Colab so that we can access files like datasets and save trained models

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **List Files in Google Drive**
This command is used to display all files and folders present in the specified directory in Google Drive.

In [ ]:
!ls /content/drive/MyDrive

SignLanguageProject  sign_language_training.ipynb


# **Load Dataset A-Z**

In [ ]:
import tensorflow as tf

data_dir = "/content/drive/MyDrive/SignLanguageProject/dataset/asl_alphabet_train"

img_size = 64
batch_size = 32

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_size, img_size),
    batch_size=batch_size
)

val_data = tf.keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_size, img_size),
    batch_size=batch_size
)

class_names = train_data.class_names
print("Classes:", class_names)

Found 28156 files belonging to 29 classes.
Using 22525 files for training.
Found 28156 files belonging to 29 classes.
Using 5631 files for validation.
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


## **Create Small Dataset (A–E Classes Only)**

In [ ]:
import shutil, os

base_path = "/content/drive/MyDrive/SignLanguageProject/dataset/asl_alphabet_train"
new_path = "/content/drive/MyDrive/SignLanguageProject/dataset/small_dataset"
classes = ['A', 'B', 'C', 'D', 'E']

os.makedirs(new_path, exist_ok=True)

for cls in classes:
    src = os.path.join(base_path, cls)
    dst = os.path.join(new_path, cls)

    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Copied {len(os.listdir(src))} images for class {cls}")
    else:
        print(f"Source folder not found: {src}")

# Verify
for cls in classes:
    cls_path = os.path.join(new_path, cls)
    print(cls, "has", len(os.listdir(cls_path)), "images")

Copied 3000 images for class A
Copied 3000 images for class B
Copied 3000 images for class C
Copied 3000 images for class D
Copied 3000 images for class E
A has 3000 images
B has 3000 images
C has 3000 images
D has 3000 images
E has 3000 images


# **Load Small Dataset for Training and Validation**
This code loads and prepares the dataset by resizing images and splitting them into training and validation sets.

In [ ]:
import tensorflow as tf

data_dir = "/content/drive/MyDrive/SignLanguageProject/dataset/small_dataset"

# Load training and validation datasets
train_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,  # 20% for validation
    subset="training",
    seed=123,
    image_size=(64, 64),
    batch_size=32
)

val_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(64, 64),
    batch_size=32
)

Found 15000 files belonging to 5 classes.
Using 12000 files for training.
Found 15000 files belonging to 5 classes.
Using 3000 files for validation.


# **Optimize Dataset for Faster Training**
This code optimizes data loading to make training faster and more efficient.

In [ ]:
import tensorflow as tf

AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_data = val_data.cache().prefetch(buffer_size=AUTOTUNE)

print("Datasets optimized for training")

Datasets optimized for training


# **Build the CNN Model**
The model takes an image as input, extracts features using convolution layers, and finally predicts the correct alphabet using a softmax output layer.

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(64, 64, 3)),  # Takes image of size 64×64 with 3 color channels (RGB)
    layers.Rescaling(1./255),         # normalize pixel values from 0–255 to 0–1 for better training
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(5, activation='softmax')  # 5 classes: A-E
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,845 (2.61 MB)

 Trainable params: 683,845 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

# **Train the Model**

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 134s 357ms/step - accuracy: 1.0000 - loss: 2.6244e-04 - val_accuracy: 1.0000 - val_loss: 4.2614e-04
Epoch 2/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 125s 333ms/step - accuracy: 1.0000 - loss: 1.2045e-04 - val_accuracy: 1.0000 - val_loss: 2.7814e-04
Epoch 3/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 124s 331ms/step - accuracy: 1.0000 - loss: 7.6338e-05 - val_accuracy: 1.0000 - val_loss: 2.1005e-04
Epoch 4/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 124s 331ms/step - accuracy: 1.0000 - loss: 5.2609e-05 - val_accuracy: 1.0000 - val_loss: 1.6377e-04
Epoch 5/5
375/375 ━━━━━━━━━━━━━━━━━━━━ 124s 329ms/step - accuracy: 1.0000 - loss: 3.7688e-05 - val_accuracy: 1.0000 - val_loss: 1.2008e-04


# **Save Model**

In [ ]:
model.save("/content/drive/MyDrive/SignLanguageProject/model/sign_model.h5")

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/SignLanguageProject/model/sign_model.h5')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>